# Phase 1: Data Preprocessing & Cleaning
Health data from wearables and medical devices is notoriously noisy and misaligned. Your models will fail if you don't handle the temporal nature of this data correctly.

## Cleaning Steps & Reasoning:

1. Datetime Conversion & Indexing: Convert the time column to a proper datetime object and set it as the index. Time-series analysis requires a continuous time index.
1. Frequency Resampling: CGM sensors (FreeStyle Libre) typically record every 5 or 15 minutes, while Fitbit HR might be per minute, and meals/insulin are point-in-time events. You must resample the entire dataset to a uniform frequency (e.g., 5-minute intervals) to align the features.
1. Contextual Imputation: 
    * bolus_volume_delivered and `carb_input`: Fill NaN with 0. If there's no record, the patient didn't eat or take rapid-acting insulin.
    * basal_rate: Forward-fill (ffill). Background insulin runs continuously until changed.
    * `glucose` and `heart_rate`: Use linear interpolation for short gaps (e.g., < 30 mins) to maintain the trend. Drop rows for massive gaps.
    * Outlier Removal: Filter out physiological impossibilities (e.g., `glucose` < 39 mg/dL or > 400 mg/dL, heart rate > 220 bpm).

In [3]:
import pandas as pd

def preprocess_t1dm_data(df):
    # 1. Datetime alignment
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time').sort_index()
    
    # 2 & 3. Resample to 5-minute intervals and apply contextual imputation
    # For continuous metrics, we take the mean. For point events (carbs/bolus), we sum them.
    resampled = df.resample('5min').agg({
        'glucose': 'mean',
        'calories': 'sum',
        'heart_rate': 'mean',
        'steps': 'sum',
        'basal_rate': 'last', # Take the last known basal rate
        'bolus_volume_delivered': 'sum',
        'carb_input': 'sum'
    })
    
    resampled['bolus_volume_delivered'] = resampled['bolus_volume_delivered'].fillna(0)
    resampled['carb_input'] = resampled['carb_input'].fillna(0)
    resampled['basal_rate'] = resampled['basal_rate'].ffill()
    
    # Interpolate small gaps (limit to 6 periods = 30 mins)
    resampled['glucose'] = resampled['glucose'].interpolate(method='linear', limit=6)
    resampled['heart_rate'] = resampled['heart_rate'].interpolate(method='linear', limit=6)
    
    # 4. Drop remaining extreme outliers/NaNs
    resampled = resampled.dropna(subset=['glucose']) 
    return resampled

## Read a file and apply clean up

In [4]:
def read_csv_file(file_path):
    raw_data = pd.read_csv(file_path, header=0, delimiter=';')
    return raw_data

In [7]:
raw_data = read_csv_file('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')
raw_data

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2018-06-13T22:45:00,86.000000,6.21384,65.390244,0.0,0.108333,0.0,0.0
1,2018-06-13T22:50:00,82.000000,6.33568,64.269231,0.0,0.108333,0.0,0.0
2,2018-06-13T22:55:00,78.000000,6.57936,68.162791,0.0,0.108333,8.0,0.0
3,2018-06-13T23:00:00,74.000000,6.09200,63.638655,0.0,0.075000,0.0,0.0
4,2018-06-13T23:05:00,76.666667,6.09200,65.486957,0.0,0.075000,0.0,0.0
...,...,...,...,...,...,...,...,...
3176,2018-06-24T23:25:00,104.000000,6.21384,65.517241,0.0,0.075000,0.0,0.0
3177,2018-06-24T23:30:00,107.000000,6.45752,66.641026,0.0,0.075000,0.0,0.0
3178,2018-06-24T23:35:00,110.666667,6.45752,65.545455,0.0,0.075000,0.0,65.0
3179,2018-06-24T23:40:00,114.333333,10.96560,68.589744,31.0,0.075000,7.5,0.0


In [8]:
def read_and_preprocess_data(file_path):
    raw_data = read_csv_file(file_path)
    processed_data = preprocess_t1dm_data(raw_data)
    return processed_data

In [9]:
pre_processed_data = read_and_preprocess_data('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')
pre_processed_data

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
time,,,,,,,
2018-06-13 22:45:00,86.000000,6.21384,65.390244,0.0,0.108333,0.0,0.0
2018-06-13 22:50:00,82.000000,6.33568,64.269231,0.0,0.108333,0.0,0.0
2018-06-13 22:55:00,78.000000,6.57936,68.162791,0.0,0.108333,8.0,0.0
2018-06-13 23:00:00,74.000000,6.09200,63.638655,0.0,0.075000,0.0,0.0
2018-06-13 23:05:00,76.666667,6.09200,65.486957,0.0,0.075000,0.0,0.0
...,...,...,...,...,...,...,...
2018-06-24 23:25:00,104.000000,6.21384,65.517241,0.0,0.075000,0.0,0.0
2018-06-24 23:30:00,107.000000,6.45752,66.641026,0.0,0.075000,0.0,0.0
2018-06-24 23:35:00,110.666667,6.45752,65.545455,0.0,0.075000,0.0,65.0


In [19]:
preprocessed_data = read_and_preprocess_data('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')
preprocessed_data

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
time,,,,,,,
2018-06-13 22:45:00,86.000000,6.21384,65.390244,0.0,0.108333,0.0,0.0
2018-06-13 22:50:00,82.000000,6.33568,64.269231,0.0,0.108333,0.0,0.0
2018-06-13 22:55:00,78.000000,6.57936,68.162791,0.0,0.108333,8.0,0.0
2018-06-13 23:00:00,74.000000,6.09200,63.638655,0.0,0.075000,0.0,0.0
2018-06-13 23:05:00,76.666667,6.09200,65.486957,0.0,0.075000,0.0,0.0
...,...,...,...,...,...,...,...
2018-06-24 23:25:00,104.000000,6.21384,65.517241,0.0,0.075000,0.0,0.0
2018-06-24 23:30:00,107.000000,6.45752,66.641026,0.0,0.075000,0.0,0.0
2018-06-24 23:35:00,110.666667,6.45752,65.545455,0.0,0.075000,0.0,65.0
